# 🧪 Lab 07 — Parquet Orbit: Spatial Meaning Survives Disk 🌌🛰️

Spark could store geometry-shaped bytes in Parquet long before native spatial types existed.

That was easy:

```text
WKB
 ↓
BINARY
 ↓
Parquet BYTE_ARRAY
```

The bytes survived.

The meaning did not necessarily survive with them.

This lab tests the new boundary:

```text
Spark GEOMETRY(4326)
        ↓
Parquet physical bytes
        +
native spatial logical type
        +
CRS information
        ↓
Spark GEOMETRY(4326)
```

We will inspect the **actual Parquet footer** instead of assuming that “native” means whatever we hope it means.

### 🎯 Mission objectives

We will prove or observe that:

- the same WKB can be persisted as ordinary Spark `BINARY` or native `GEOMETRY(4326)`;
- both use a binary Parquet physical payload;
- only the native spatial file carries a Parquet spatial logical annotation;
- the native footer exposes CRS information associated with that spatial annotation;
- Spark reconstructs `BINARY` as `BINARY` and native Geometry as `GEOMETRY(4326)`;
- the WKB payload itself survives both routes;
- Parquet key/value metadata can be inspected separately from native logical types;
- a GeoParquet-style `geo` metadata key is **observed**, not assumed;
- optional geospatial/bounding-box statistics are **observed**, not assumed;
- `GEOMETRY(ANY)` hits the fixed-SRID persistence boundary.

> **Critical distinction:** Parquet's native spatial logical types, GeoParquet conventions, and optional geospatial statistics are related pieces of the ecosystem. They are not synonyms.

## 0 — Pre-flight checks 🛰️

Target runtime:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

Stock Spark only.

No GeoPandas. No Sedona. No `parquet-tools`.

For footer inspection we use the **Parquet Java classes already bundled with Spark**, which lets the notebook inspect the file Spark actually wrote without depending on an external CLI.

> `parquet-mr` calls the primitive physical type `BINARY`; in the Parquet format specification this corresponds to the `BYTE_ARRAY` physical type.

In [1]:
import sys, json, struct, shutil, tempfile, warnings, re
from pathlib import Path


warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-07-parquet-orbit-spatial-meaning-survives-disk")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")

fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert pyspark.__version__ == "4.2.0"
assert spark.version == "4.2.0"
assert int(java_version.split(".")[0]) >= 17
assert fingerprint["geospatial_enabled"].lower() == "true"

workdir = Path(tempfile.mkdtemp(prefix="spark42_parquet_orbit_")).resolve()
binary_dir = workdir / "ordinary_binary"
geometry_dir = workdir / "native_geometry_4326"
any_dir = workdir / "geometry_any_should_fail"

def spark_uri(path: Path) -> str:
    return path.resolve().as_uri()

print(f"\n🛰️ Orbit directory: {workdir}")
print("✅ Parquet observation deck online.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "geospatial_enabled": "true"
}

🛰️ Orbit directory: C:\Users\angel.alvarez\AppData\Local\Temp\spark42_parquet_orbit_iv0glcv5
✅ Parquet observation deck online.


# 1 — Same WKB, Two Contracts 🧬

We create a tiny deterministic dataset of points around Spain.

Each row starts with WKB.

Then we fork reality:

```text
same WKB
   │
   ├── payload: BINARY
   │
   └── geom: GEOMETRY(4326)
```

If the native feature is meaningful at the storage boundary, those two DataFrames should not produce equivalent Parquet schemas even though the coordinate payload is the same.

In [2]:
def wkb_point(x, y):
    return struct.pack("<BIdd", 1, 1, float(x), float(y))

places = [
    (1, "Madrid",    -3.7038, 40.4168),
    (2, "Barcelona",  2.1734, 41.3851),
    (3, "Seville",   -5.9845, 37.3891),
    (4, "Valencia",  -0.3763, 39.4699),
]

rows = [
    (id_, name, x, y, wkb_point(x, y))
    for id_, name, x, y in places
]

source = spark.createDataFrame(
    rows,
    ["id", "name", "x_reference", "y_reference", "wkb"],
)

binary_df = source.select(
    "id",
    "name",
    F.col("wkb").alias("geom"),
)

geometry_df = source.select(
    "id",
    "name",
    F.st_geomfromwkb("wkb", 4326).alias("geom"),
)

binary_type = binary_df.schema["geom"].dataType.simpleString()
geometry_type = geometry_df.schema["geom"].dataType.simpleString()

print("🧬 SAME PAYLOAD, TWO SPARK CONTRACTS")
print(f"  ├─ ordinary column : {binary_type}")
print(f"  └─ native column   : {geometry_type}")

print("\n📦 Ordinary BINARY schema")
binary_df.printSchema()

print("\n🌍 Native Geometry schema")
geometry_df.printSchema()

assert binary_type.lower() == "binary"
assert geometry_type.lower() == "geometry(4326)"

lab_results = {
    "source_rows": len(places),
    "binary_spark_type_before_write": binary_type,
    "geometry_spark_type_before_write": geometry_type,
}

🧬 SAME PAYLOAD, TWO SPARK CONTRACTS
  ├─ ordinary column : binary
  └─ native column   : geometry(4326)

📦 Ordinary BINARY schema
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- geom: binary (nullable = true)


🌍 Native Geometry schema
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- geom: geometry(4326) (nullable = true)



# 2 — Enter the Wormhole: Write Both to Parquet 🌌

We write each DataFrame to its own dataset.

The experiment deliberately uses one output part file per dataset so footer inspection stays readable.

This is not a performance recommendation.

It is an autopsy table.

In [3]:
binary_df.coalesce(1).write.mode("overwrite").parquet(spark_uri(binary_dir))
geometry_df.coalesce(1).write.mode("overwrite").parquet(spark_uri(geometry_dir))

def parquet_part_files(directory: Path):
    return sorted(
        p for p in directory.rglob("*.parquet")
        if p.is_file()
    )

binary_files = parquet_part_files(binary_dir)
geometry_files = parquet_part_files(geometry_dir)

print("🌌 PARQUET FILES CREATED")
print(f"  ├─ BINARY   : {[p.name for p in binary_files]}")
print(f"  └─ GEOMETRY : {[p.name for p in geometry_files]}")

assert len(binary_files) == 1
assert len(geometry_files) == 1

lab_results.update({
    "binary_file": str(binary_files[0]),
    "geometry_file": str(geometry_files[0]),
})

🌌 PARQUET FILES CREATED
  ├─ BINARY   : ['part-00000-313fb7cb-3f49-4f92-99fe-b3ce0b0dc4d9-c000.snappy.parquet']
  └─ GEOMETRY : ['part-00000-137bbd63-11b1-4127-9945-022cbabf43a1-c000.snappy.parquet']


# 3 — Open the Parquet Footer 🔬📦

Now we stop asking Spark what its DataFrame schema was.

We inspect the **Parquet file itself**.

For each top-level column we record:

```text
physical primitive type
logical annotation
logical annotation class
CRS-related getters, if exposed
```

We also record the raw Parquet schema text and file-level key/value metadata.

This is where “the bytes survived” and “the spatial contract survived” become visibly different claims.

In [4]:
jvm = spark._jvm
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

def java_map_to_dict(jmap):
    result = {}
    it = jmap.entrySet().iterator()
    while it.hasNext():
        entry = it.next()
        result[str(entry.getKey())] = str(entry.getValue())
    return result

def zero_arg_method_values(obj, name_fragment):
    """
    Reflection probe: call zero-argument methods whose names contain
    name_fragment. Failures are ignored because some reflected methods
    may require arguments despite having a matching name.
    """
    values = {}
    if obj is None:
        return values

    method_names = sorted({
        str(m.getName())
        for m in obj.getClass().getMethods()
        if name_fragment.lower() in str(m.getName()).lower()
    })

    for name in method_names:
        try:
            value = getattr(obj, name)()
            if value is not None:
                values[name] = str(value)
        except Exception:
            pass

    return values

def inspect_parquet_footer(parquet_file: Path):
    PathCls = jvm.org.apache.hadoop.fs.Path
    HadoopInputFile = jvm.org.apache.parquet.hadoop.util.HadoopInputFile
    ParquetFileReader = jvm.org.apache.parquet.hadoop.ParquetFileReader

    jpath = PathCls(parquet_file.resolve().as_uri())
    input_file = HadoopInputFile.fromPath(jpath, hadoop_conf)
    reader = ParquetFileReader.open(input_file)

    try:
        footer = reader.getFooter()
        file_meta = footer.getFileMetaData()
        schema = file_meta.getSchema()

        fields = []
        for field in schema.getFields():
            item = {
                "name": str(field.getName()),
                "primitive": bool(field.isPrimitive()),
            }

            if field.isPrimitive():
                primitive = field.asPrimitiveType()
                logical = primitive.getLogicalTypeAnnotation()

                item.update({
                    "physical_type": str(primitive.getPrimitiveTypeName()),
                    "logical_annotation": None if logical is None else str(logical),
                    "logical_class": None if logical is None else str(logical.getClass().getName()),
                    "crs_getters": zero_arg_method_values(logical, "crs"),
                })

            fields.append(item)

        kv = java_map_to_dict(file_meta.getKeyValueMetaData())

        return {
            "file": str(parquet_file),
            "schema_text": str(schema),
            "fields": fields,
            "key_value_metadata": kv,
            "row_groups": int(len(footer.getBlocks())),
        }
    finally:
        reader.close()

binary_footer = inspect_parquet_footer(binary_files[0])
geometry_footer = inspect_parquet_footer(geometry_files[0])

print("📦 ORDINARY BINARY — PARQUET SCHEMA")
print(binary_footer["schema_text"])

print("\n🌍 NATIVE GEOMETRY — PARQUET SCHEMA")
print(geometry_footer["schema_text"])

print("\n🔬 COLUMN FORENSICS")
print("\nBINARY dataset:")
print(json.dumps(binary_footer["fields"], indent=2))

print("\nGEOMETRY dataset:")
print(json.dumps(geometry_footer["fields"], indent=2))

lab_results.update({
    "binary_footer": binary_footer,
    "geometry_footer": geometry_footer,
})

📦 ORDINARY BINARY — PARQUET SCHEMA
message spark_schema {
  optional int64 id;
  optional binary name (STRING);
  optional binary geom;
}


🌍 NATIVE GEOMETRY — PARQUET SCHEMA
message spark_schema {
  optional int64 id;
  optional binary name (STRING);
  optional binary geom (GEOMETRY(OGC:CRS84));
}


🔬 COLUMN FORENSICS

BINARY dataset:
[
  {
    "name": "id",
    "primitive": true,
    "physical_type": "INT64",
    "logical_annotation": null,
    "logical_class": null,
    "crs_getters": {}
  },
  {
    "name": "name",
    "primitive": true,
    "physical_type": "BINARY",
    "logical_annotation": "STRING",
    "logical_class": "org.apache.parquet.schema.LogicalTypeAnnotation$StringLogicalTypeAnnotation",
    "crs_getters": {}
  },
  {
    "name": "geom",
    "primitive": true,
    "physical_type": "BINARY",
    "logical_annotation": null,
    "logical_class": null,
    "crs_getters": {}
  }
]

GEOMETRY dataset:
[
  {
    "name": "id",
    "primitive": true,
    "physical_type": "INT64

# 4 — Physical Bytes vs Logical Meaning 🧠

The column named `geom` is the important one.

We compare:

```text
ordinary BINARY file
native GEOMETRY file
```

At the physical level we expect a binary payload in both cases.

The crucial difference should live in the **logical annotation**.

Let's make that comparison explicit from the footer evidence we just captured.

In [5]:
def field_by_name(footer_info, name):
    return next(f for f in footer_info["fields"] if f["name"] == name)

binary_geom_footer = field_by_name(binary_footer, "geom")
native_geom_footer = field_by_name(geometry_footer, "geom")

print("🧠 FILE-FORMAT CONTRACT COMPARISON")
print(f"  ├─ ordinary physical type : {binary_geom_footer.get('physical_type')}")
print(f"  ├─ ordinary logical type  : {binary_geom_footer.get('logical_annotation')}")
print(f"  ├─ native physical type   : {native_geom_footer.get('physical_type')}")
print(f"  ├─ native logical type    : {native_geom_footer.get('logical_annotation')}")
print(f"  └─ native logical class   : {native_geom_footer.get('logical_class')}")

binary_physical = str(binary_geom_footer.get("physical_type", "")).upper()
native_physical = str(native_geom_footer.get("physical_type", "")).upper()
native_logical_text = str(native_geom_footer.get("logical_annotation") or "")
native_logical_class = str(native_geom_footer.get("logical_class") or "")

assert binary_physical == "BINARY"
assert native_physical == "BINARY"
assert binary_geom_footer.get("logical_annotation") is None
assert (
    "geometry" in native_logical_text.lower()
    or "geometry" in native_logical_class.lower()
), native_geom_footer

lab_results.update({
    "binary_physical_type": binary_physical,
    "native_physical_type": native_physical,
    "binary_logical_annotation": binary_geom_footer.get("logical_annotation"),
    "native_logical_annotation": native_geom_footer.get("logical_annotation"),
    "native_logical_class": native_geom_footer.get("logical_class"),
})

🧠 FILE-FORMAT CONTRACT COMPARISON
  ├─ ordinary physical type : BINARY
  ├─ ordinary logical type  : None
  ├─ native physical type   : BINARY
  ├─ native logical type    : GEOMETRY(OGC:CRS84)
  └─ native logical class   : org.apache.parquet.schema.LogicalTypeAnnotation$GeometryLogicalTypeAnnotation


## The wormhole in one picture

```text
SAME KIND OF WKB PAYLOAD
          │
          ├─────────────────────────────┐
          │                             │
 ordinary BINARY                  native GEOMETRY(4326)
          │                             │
          ▼                             ▼
 Parquet binary primitive       Parquet binary primitive
 logical annotation: none       logical annotation: GEOMETRY(...)
          │                             │
          ▼                             ▼
 "these are bytes"              "these bytes are spatial"
```

That is the file-format breakthrough.

The physical bytes were never the hard part.

**Preserving the contract was.**

## 📦 Surprise Cargo: Spark Wrote a Native Bbox

The footer autopsy revealed more than just a `GEOMETRY` annotation.

This Spark 4.2 writer also emitted native:

```text
GeospatialStatistics
└── BoundingBox
    ├── xMin = -5.9845
    ├── xMax =  2.1734
    ├── yMin = 37.3891
    └── yMax = 41.3851
```

And the dedicated cross-examination later in the notebook proves those four values match the source geometries exactly.

> **The storage layer did not merely remember “this is spatial.” It also wrote a coarse spatial envelope.**

# 5 — Where Is the CRS? 🌍🧭

Native Parquet spatial logical types can carry CRS information.

Instead of hard-coding how a particular `parquet-mr` version chooses to print the annotation, we inspect:

1. the logical annotation text;
2. its Java class;
3. any CRS-related zero-argument getters exposed by that annotation object;
4. the complete schema text.

For Spark's SRID `4326`, the file should contain enough native spatial information for Spark to reconstruct the fixed spatial type on readback.

We record the exact footer representation this runtime produced.

In [6]:
crs_probe = {
    "logical_annotation": native_geom_footer.get("logical_annotation"),
    "logical_class": native_geom_footer.get("logical_class"),
    "crs_getters": native_geom_footer.get("crs_getters"),
    "schema_text": geometry_footer["schema_text"],
}

print("🌍 CRS FOOTER EVIDENCE")
print(json.dumps(crs_probe, indent=2))

# We do not hard-code one textual rendering of the CRS annotation.
# The stronger end-to-end assertion comes in the readback section:
# Spark must reconstruct GEOMETRY(4326), not merely BINARY.
lab_results["crs_probe"] = crs_probe

🌍 CRS FOOTER EVIDENCE
{
  "logical_annotation": "GEOMETRY(OGC:CRS84)",
  "logical_class": "org.apache.parquet.schema.LogicalTypeAnnotation$GeometryLogicalTypeAnnotation",
  "crs_getters": {
    "getCrs": "OGC:CRS84"
  },
  "schema_text": "message spark_schema {\n  optional int64 id;\n  optional binary name (STRING);\n  optional binary geom (GEOMETRY(OGC:CRS84));\n}\n"
}


## 🧬 Three Layers, Three Different Claims

```text
1. PHYSICAL PAYLOAD
   BINARY / BYTE_ARRAY
   └── WKB bytes


2. PARQUET LOGICAL TYPE
   GEOMETRY(OGC:CRS84)
   └── "these bytes are spatial, with this CRS"


3. FILE-LEVEL INTEROPERABILITY CONVENTIONS
   GeoParquet metadata such as `geo = {...}`
   └── broader ecosystem conventions
```

These layers can coexist, but they are **not interchangeable statements**.

That distinction is exactly why we inspect the actual file instead of declaring victory after `write.parquet(...)`.

# 6 — GeoParquet: Related, Not Automatically Identical 🗺️📦

Historically, GeoParquet commonly uses file-level Parquet key/value metadata such as a `geo` JSON entry to describe spatial columns and conventions.

Native Parquet spatial logical types move spatial typing into Parquet's own schema system.

Those are related ideas, but they are not the same metadata mechanism.

So we ask the files directly:

```text
What file-level metadata keys exist?
Is there a `geo` key?
```

No assumption either way.

In [7]:
def interesting_kv_metadata(footer):
    kv = footer["key_value_metadata"]
    return {
        "keys": sorted(kv.keys()),
        "has_geo_key": "geo" in kv,
        "geo_value": kv.get("geo"),
        "spark_schema_metadata_present": "org.apache.spark.sql.parquet.row.metadata" in kv,
    }

binary_kv = interesting_kv_metadata(binary_footer)
geometry_kv = interesting_kv_metadata(geometry_footer)

print("🗺️ FILE-LEVEL KEY/VALUE METADATA")
print("\nBINARY:")
print(json.dumps(binary_kv, indent=2))

print("\nNATIVE GEOMETRY:")
print(json.dumps(geometry_kv, indent=2))

lab_results.update({
    "binary_kv": binary_kv,
    "geometry_kv": geometry_kv,
})

🗺️ FILE-LEVEL KEY/VALUE METADATA

BINARY:
{
  "keys": [
    "org.apache.spark.sql.parquet.row.metadata",
    "org.apache.spark.version"
  ],
  "has_geo_key": false,
  "geo_value": null,
  "spark_schema_metadata_present": true
}

NATIVE GEOMETRY:
{
  "keys": [
    "org.apache.spark.sql.parquet.row.metadata",
    "org.apache.spark.version"
  ],
  "has_geo_key": false,
  "geo_value": null,
  "spark_schema_metadata_present": true
}


## Why this matters

This run gives us a clean forensic result:

```text
native Parquet `GEOMETRY` annotation  → ✅ present
GeoParquet-style `geo` metadata key   → ❌ absent
```

> **Native spatial Parquet does not automatically mean GeoParquet metadata conventions were emitted. This file proves the distinction.**

The spatial contract survived through Parquet's **native schema mechanism** even though the GeoParquet-style `geo` key was not present.

# 7 — Bounding Boxes: Did This Writer Actually Emit Them? 📦🔭

Parquet's geospatial specification can support optional geospatial statistics such as bounding boxes.

Those statistics could become powerful pruning material:

```text
bounding boxes do not overlap
            ↓
geometries cannot intersect
            ↓
skip irrelevant work
```

But there are two separate questions:

```text
Can the Parquet format represent this?
Does this Spark writer emit it here?
```

We are testing the second one.

Because the exact Java API surface can evolve, this probe uses reflection on the actual row-group column metadata and looks for geospatial/bounding-box methods.

**No bbox will be fabricated if the writer did not produce one.**

In [8]:
def inspect_geo_statistics_api(parquet_file: Path):
    PathCls = jvm.org.apache.hadoop.fs.Path
    HadoopInputFile = jvm.org.apache.parquet.hadoop.util.HadoopInputFile
    ParquetFileReader = jvm.org.apache.parquet.hadoop.ParquetFileReader

    jpath = PathCls(parquet_file.resolve().as_uri())
    input_file = HadoopInputFile.fromPath(jpath, hadoop_conf)
    reader = ParquetFileReader.open(input_file)

    observations = []

    try:
        footer = reader.getFooter()

        for rg_index, block in enumerate(footer.getBlocks()):
            for col in block.getColumns():
                path = str(col.getPath().toDotString())

                method_names = sorted({
                    str(m.getName())
                    for m in col.getClass().getMethods()
                    if (
                        "geo" in str(m.getName()).lower()
                        or "bound" in str(m.getName()).lower()
                    )
                })

                values = {}
                for name in method_names:
                    try:
                        value = getattr(col, name)()
                        if value is not None:
                            values[name] = str(value)
                    except Exception:
                        pass

                observations.append({
                    "row_group": rg_index,
                    "column": path,
                    "candidate_methods": method_names,
                    "non_null_values": values,
                })

        return observations
    finally:
        reader.close()

geo_stats_observations = inspect_geo_statistics_api(geometry_files[0])

print("🔭 GEOSPATIAL / BOUNDING-BOX FOOTER PROBE")
for obs in geo_stats_observations:
    if obs["column"] != "geom":
        continue
    print(json.dumps(obs, indent=2))

geom_geo_stats = [
    obs for obs in geo_stats_observations
    if obs["column"] == "geom"
]

geo_stats_non_null = any(
    bool(obs["non_null_values"])
    for obs in geom_geo_stats
)

print(f"\n📡 Non-null geospatial/bbox metadata exposed for geom? {geo_stats_non_null}")

lab_results.update({
    "geo_stats_observations": geom_geo_stats,
    "geo_stats_non_null": geo_stats_non_null,
})

🔭 GEOSPATIAL / BOUNDING-BOX FOOTER PROBE
{
  "row_group": 0,
  "column": "geom",
  "candidate_methods": [
    "getDictionaryPageOffset",
    "getFirstDataPageOffset",
    "getGeospatialStatistics"
  ],
  "non_null_values": {
    "getDictionaryPageOffset": "0",
    "getFirstDataPageOffset": "137",
    "getGeospatialStatistics": "GeospatialStatistics{boundingBox=BoundingBox{xMin=-5.9845, xMax=2.1734, yMin=37.3891, yMax=41.3851, zMin=NaN, zMax=NaN, mMin=NaN, mMax=NaN}, geospatialTypes=GeospatialTypes{types=[Point (XY)]}}"
  }
}

📡 Non-null geospatial/bbox metadata exposed for geom? True


## 🎯 Bounding Box Cross-Examination

The writer exposed native `GeospatialStatistics`.

Now we make the evidence harder to hand-wave away.

For our four source points, the expected envelope is deterministic:

```text
xMin = min(longitudes) = -5.9845
xMax = max(longitudes) =  2.1734
yMin = min(latitudes)  = 37.3891
yMax = max(latitudes)  = 41.3851
```

The next cell extracts the bbox from the actual Parquet row-group metadata and asserts that every bound matches the source data.

In [9]:
expected_bbox = {
    "xMin": min(x for _, _, x, _ in places),
    "xMax": max(x for _, _, x, _ in places),
    "yMin": min(y for _, _, _, y in places),
    "yMax": max(y for _, _, _, y in places),
}

def extract_bbox_from_observations(observations):
    """
    Parse xMin/xMax/yMin/yMax from the actual GeospatialStatistics string
    returned by parquet-mr for the geom column.
    """
    candidates = []
    for obs in observations:
        for value in obs.get("non_null_values", {}).values():
            text = str(value)
            if "BoundingBox" not in text:
                continue
            match = re.search(
                r"xMin=([-+0-9.Ee]+).*?"
                r"xMax=([-+0-9.Ee]+).*?"
                r"yMin=([-+0-9.Ee]+).*?"
                r"yMax=([-+0-9.Ee]+)",
                text,
                flags=re.S,
            )
            if match:
                candidates.append({
                    "xMin": float(match.group(1)),
                    "xMax": float(match.group(2)),
                    "yMin": float(match.group(3)),
                    "yMax": float(match.group(4)),
                })

    if not candidates:
        raise AssertionError(
            "No BoundingBox could be extracted from the actual GeospatialStatistics output."
        )

    if len(candidates) != 1:
        print(f"ℹ️ Found {len(candidates)} bbox candidates; using the first row-group bbox.")

    return candidates[0]

observed_bbox = extract_bbox_from_observations(geom_geo_stats)

print("🎯 BOUNDING BOX CROSS-EXAMINATION")
print(f"  ├─ expected : {expected_bbox}")
print(f"  └─ observed : {observed_bbox}")

for key in expected_bbox:
    delta = abs(observed_bbox[key] - expected_bbox[key])
    assert delta <= 1e-12, (
        key,
        expected_bbox[key],
        observed_bbox[key],
        delta,
    )

print("\n✅ Native Parquet bbox matches the source geometries exactly.")

lab_results.update({
    "expected_bbox": expected_bbox,
    "observed_bbox": observed_bbox,
    "bbox_matches_source": True,
})

🎯 BOUNDING BOX CROSS-EXAMINATION
  ├─ expected : {'xMin': -5.9845, 'xMax': 2.1734, 'yMin': 37.3891, 'yMax': 41.3851}
  └─ observed : {'xMin': -5.9845, 'xMax': 2.1734, 'yMin': 37.3891, 'yMax': 41.3851}

✅ Native Parquet bbox matches the source geometries exactly.


## Read this result carefully

The experiment proved something concrete:

```text
Spark wrote native GeospatialStatistics  ✅
Spark wrote the correct bounding box      ✅
```

It did **not** test whether Spark uses that bbox to skip row groups for spatial predicates.

> **We proved Spark wrote the bbox. We did not prove Spark uses it to skip data.**

# 8 — Come Back Through the Wormhole 🔄🌌

Now read both datasets back with Spark.

If the spatial contract truly survived disk, the schemas should reconstruct differently:

```text
ordinary file → BINARY
native file   → GEOMETRY(4326)
```

Then we compare the actual WKB payload row by row.

Same shape bytes.

Different surviving contract.

In [10]:
binary_back = spark.read.parquet(spark_uri(binary_dir))
geometry_back = spark.read.parquet(spark_uri(geometry_dir))

binary_back_type = binary_back.schema["geom"].dataType.simpleString()
geometry_back_type = geometry_back.schema["geom"].dataType.simpleString()

print("🔄 READBACK SCHEMAS")
print(f"  ├─ ordinary file → {binary_back_type}")
print(f"  └─ native file   → {geometry_back_type}")

binary_back_rows = {
    row["id"]: row["geom"]
    for row in binary_back.select("id", "geom").collect()
}

geometry_back_rows = {
    row["id"]: row["wkb"]
    for row in (
        geometry_back
        .select(
            "id",
            F.st_asbinary("geom").alias("wkb"),
        )
        .collect()
    )
}

source_rows = {
    row["id"]: row["wkb"]
    for row in source.select("id", "wkb").collect()
}

binary_wkb_exact = binary_back_rows == source_rows
geometry_wkb_exact = geometry_back_rows == source_rows

native_srids = [
    row["srid"]
    for row in (
        geometry_back
        .select(F.st_srid("geom").alias("srid"))
        .distinct()
        .collect()
    )
]

print("\n🧬 PAYLOAD + CONTRACT ROUND TRIP")
print(f"  ├─ BINARY WKB exact      : {binary_wkb_exact}")
print(f"  ├─ GEOMETRY WKB exact    : {geometry_wkb_exact}")
print(f"  └─ native SRIDs          : {native_srids}")

assert binary_back_type.lower() == "binary"
assert geometry_back_type.lower() == "geometry(4326)"
assert binary_wkb_exact
assert geometry_wkb_exact
assert native_srids == [4326]

lab_results.update({
    "binary_spark_type_after_read": binary_back_type,
    "geometry_spark_type_after_read": geometry_back_type,
    "binary_wkb_exact": binary_wkb_exact,
    "geometry_wkb_exact": geometry_wkb_exact,
    "native_srids_after_read": native_srids,
})

🔄 READBACK SCHEMAS
  ├─ ordinary file → binary
  └─ native file   → geometry(4326)

🧬 PAYLOAD + CONTRACT ROUND TRIP
  ├─ BINARY WKB exact      : True
  ├─ GEOMETRY WKB exact    : True
  └─ native SRIDs          : [4326]


# 9 — `GEOMETRY(ANY)` Tries to Board the Ship 🚫🌌

Query-time mixed SRIDs are useful:

```text
GEOMETRY(4326)
GEOMETRY(3857)
       ↓
GEOMETRY(ANY)
```

But a persistent spatial file needs a stable spatial contract.

We create a real `GEOMETRY(ANY)` column and attempt to write it to Parquet.

The important thing to capture is not a giant stack trace.

It is simply:

```text
Did the writer accept the mixed-SRID spatial type?
If not, what Spark condition rejected it?
```

In [11]:
g4326 = (
    spark.createDataFrame(
        [(1, wkb_point(-3.7038, 40.4168))],
        ["id", "wkb"],
    )
    .select("id", F.st_geomfromwkb("wkb", 4326).alias("geom"))
)

g3857 = (
    spark.createDataFrame(
        [(2, wkb_point(-412305.13, 4926696.67))],
        ["id", "wkb"],
    )
    .select("id", F.st_geomfromwkb("wkb", 3857).alias("geom"))
)

mixed = g4326.unionByName(g3857)
mixed_type = mixed.schema["geom"].dataType.simpleString()

def compact_exception(exc):
    lines = [line.strip() for line in str(exc).splitlines() if line.strip()]
    first = lines[0] if lines else repr(exc)

    condition = None
    getter = getattr(exc, "getCondition", None)
    if callable(getter):
        try:
            condition = getter()
        except Exception:
            condition = None

    if condition is None:
        m = re.search(r"\[([A-Z0-9_.]+)\]", first)
        if m:
            condition = m.group(1)

    return {
        "type": type(exc).__name__,
        "condition": condition,
        "message": first,
    }

if any_dir.exists():
    shutil.rmtree(any_dir)

any_write_error = None

# Expected failure: keep the notebook readable.
spark.sparkContext.setLogLevel("OFF")
try:
    mixed.write.mode("overwrite").parquet(spark_uri(any_dir))
except Exception as exc:
    any_write_error = compact_exception(exc)
finally:
    spark.sparkContext.setLogLevel("ERROR")

print("🚫 MIXED-SRID PERSISTENCE PROBE")
print(f"  ├─ input type      : {mixed_type}")
print(f"  ├─ write succeeded : {any_write_error is None}")
print(f"  └─ error           : {json.dumps(any_write_error, indent=2) if any_write_error else None}")

assert mixed_type.lower() == "geometry(any)"
assert any_write_error is not None

lab_results.update({
    "mixed_type": mixed_type,
    "any_write_succeeded": any_write_error is None,
    "any_write_error": any_write_error,
})

🚫 MIXED-SRID PERSISTENCE PROBE
  ├─ input type      : geometry(any)
  ├─ write succeeded : False
  └─ error           : {
  "type": "AnalysisException",
  "condition": "UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE",
  "message": "[UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE] The Parquet datasource doesn't support the column `geom` of the type \"GEOMETRY(ANY)\". SQLSTATE: 0A000"
}


## 🚪 Persistence Border Control

```text
GEOMETRY(4326)
        │
        └── Parquet ✅


GEOMETRY(ANY)
        │
        └── Parquet 💥
            UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE
```

Query-time mixed SRIDs are useful.

Persistent native spatial storage eventually asks a less philosophical question:

> **Which CRS does this column actually use?**

# 10 — The Parquet Wormhole in One Screen 🛰️📦

```text
                          SAME WKB
                             │
             ┌───────────────┴────────────────┐
             │                                │
          BINARY                      GEOMETRY(4326)
             │                                │
             ▼                                ▼
     Parquet binary primitive         Parquet binary primitive
     no spatial logical type          GEOMETRY logical type
             │                        + CRS contract
             │                                │
             ▼                                ▼
       Spark BINARY                 Spark GEOMETRY(4326)
```

And separately:

```text
GeoParquet key/value conventions
               │
               └── inspect; do not assume

optional bbox/geospatial statistics
               │
               └── inspect; do not assume

GEOMETRY(ANY)
               │
               └── query-time multiverse, not a fixed Parquet contract
```

The fastest geometry may indeed be the geometry you never read.

But first the storage layer has to preserve enough spatial meaning for future readers and optimizers to use.

## 🩻 File Autopsy — One Screen

| Evidence | Ordinary WKB file | Native spatial file |
|---|---|---|
| Spark input type | `BINARY` | `GEOMETRY(4326)` |
| Parquet physical type | `BINARY` | `BINARY` |
| Spatial logical annotation | none | `GEOMETRY(OGC:CRS84)` |
| CRS in native annotation | none | `OGC:CRS84` |
| GeoParquet-style `geo` key | absent | absent |
| Native `GeospatialStatistics` | none observed | **present** |
| Native bounding box | none observed | **present** |
| Spark readback type | `BINARY` | `GEOMETRY(4326)` |
| WKB survived | ✅ | ✅ |
| SRID reconstructed | — | `4326` |

The physical payload stayed binary in both files.

The difference is that the native file carried **spatial meaning through the file format**.

# 📊 Post-Lab Analysis — Let the Footer Testify

The next cell builds the verdict from this run's evidence and keeps three categories separate:

```text
PROVEN
→ schema, footer, bbox, readback, persistence behavior

OBSERVED
→ metadata conventions actually present in this file

NOT CLAIMED
→ that Spark already exploits every spatial statistic for pushdown
```

In [12]:
from IPython.display import Markdown, display

native_logical = lab_results["native_logical_annotation"]
geo_key = lab_results["geometry_kv"]["has_geo_key"]
geo_stats = lab_results["geo_stats_non_null"]
any_error = lab_results["any_write_error"]

analysis = f"""
# 📊 Post-Lab Analysis: The Spatial Contract Survived the Wormhole

We wrote the same kind of WKB payload under two Spark contracts:

```text
ordinary → {lab_results['binary_spark_type_before_write']}
native   → {lab_results['geometry_spark_type_before_write']}
```

### 1. Physically, Both Columns Were Binary

The Parquet footer reported:

```text
ordinary physical → {lab_results['binary_physical_type']}
native physical   → {lab_results['native_physical_type']}
```

So native spatial support did **not** require inventing a new coordinate payload.

The WKB still lives in a binary physical column.

### 2. Logically, the Files Were Not the Same

Ordinary BINARY logical annotation:

```text
{lab_results['binary_logical_annotation']}
```

Native Geometry logical annotation:

```text
{native_logical}
```

That is the file-format distinction the section cares about.

One file stores opaque bytes.

The other stores bytes with a native spatial contract.

### 3. Spark Reconstructed the Contract After Disk

Readback types:

```text
ordinary → {lab_results['binary_spark_type_after_read']}
native   → {lab_results['geometry_spark_type_after_read']}
```

Native readback SRID:

```text
{lab_results['native_srids_after_read']}
```

WKB survived exactly through both routes:

```text
ordinary BINARY → {lab_results['binary_wkb_exact']}
native Geometry → {lab_results['geometry_wkb_exact']}
```

The shape bytes survived both.

Only the native route also reconstructed the spatial type and CRS contract.

### 4. GeoParquet Metadata Was Inspected, Not Assumed

Did the native file contain a top-level `geo` metadata key?

**{geo_key}**

Whatever this runtime observed, the conclusion stays disciplined:

> Native Parquet spatial logical types and GeoParquet metadata conventions are related, but they are not automatically the same claim.

### 5. Bounding-Box / Geospatial Statistics Were Also Inspected

Did the bundled Parquet metadata API expose a non-null geospatial/bbox value for `geom`?

**{geo_stats}**

This run goes beyond mere format capability: **the writer emitted native GeospatialStatistics for `geom`, including a bounding box.**

That is an observation about this writer and this file.

It still does **not** prove that Spark consumes that metadata for every spatial predicate, filter, or join.

**We proved Spark wrote the bbox. We did not prove Spark uses it for spatial predicate pushdown.**

### 6. The Multiverse Still Cannot Live on Disk as a Native Spatial Column

Input type:

```text
{lab_results['mixed_type']}
```

Parquet write succeeded:

**{lab_results['any_write_succeeded']}**

Observed Spark condition:

```text
{None if any_error is None else any_error['condition']}
```

So the query engine can temporarily admit mixed-SRID Geometry, while persistent native spatial storage requires a fixed contract.

> ## 🚀 Mission Verdict
> Old Spark could preserve geometry **bytes** in Parquet.
>
> Spark 4.2 can preserve a native **spatial contract** in Parquet and reconstruct it on read.
>
> That is the difference between:
>
> ```text
> BYTE_ARRAY
> ¯\\_(ツ)_/¯
> ```
>
> and:
>
> ```text
> binary payload
> + spatial logical type
> + CRS contract
> ```
>
> **The breakthrough is not that the polygon survives disk. The meaning survives with it.**
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: The Spatial Contract Survived the Wormhole

We wrote the same kind of WKB payload under two Spark contracts:

```text
ordinary → binary
native   → geometry(4326)
```

### 1. Physically, Both Columns Were Binary

The Parquet footer reported:

```text
ordinary physical → BINARY
native physical   → BINARY
```

So native spatial support did **not** require inventing a new coordinate payload.

The WKB still lives in a binary physical column.

### 2. Logically, the Files Were Not the Same

Ordinary BINARY logical annotation:

```text
None
```

Native Geometry logical annotation:

```text
GEOMETRY(OGC:CRS84)
```

That is the file-format distinction the section cares about.

One file stores opaque bytes.

The other stores bytes with a native spatial contract.

### 3. Spark Reconstructed the Contract After Disk

Readback types:

```text
ordinary → binary
native   → geometry(4326)
```

Native readback SRID:

```text
[4326]
```

WKB survived exactly through both routes:

```text
ordinary BINARY → True
native Geometry → True
```

The shape bytes survived both.

Only the native route also reconstructed the spatial type and CRS contract.

### 4. GeoParquet Metadata Was Inspected, Not Assumed

Did the native file contain a top-level `geo` metadata key?

**False**

Whatever this runtime observed, the conclusion stays disciplined:

> Native Parquet spatial logical types and GeoParquet metadata conventions are related, but they are not automatically the same claim.

### 5. Bounding-Box / Geospatial Statistics Were Also Inspected

Did the bundled Parquet metadata API expose a non-null geospatial/bbox value for `geom`?

**True**

This run goes beyond mere format capability: **the writer emitted native GeospatialStatistics for `geom`, including a bounding box.**

That is an observation about this writer and this file.

It still does **not** prove that Spark consumes that metadata for every spatial predicate, filter, or join.

**We proved Spark wrote the bbox. We did not prove Spark uses it for spatial predicate pushdown.**

### 6. The Multiverse Still Cannot Live on Disk as a Native Spatial Column

Input type:

```text
geometry(any)
```

Parquet write succeeded:

**False**

Observed Spark condition:

```text
UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE
```

So the query engine can temporarily admit mixed-SRID Geometry, while persistent native spatial storage requires a fixed contract.

> ## 🚀 Mission Verdict
> Old Spark could preserve geometry **bytes** in Parquet.
>
> Spark 4.2 can preserve a native **spatial contract** in Parquet and reconstruct it on read.
>
> That is the difference between:
>
> ```text
> BYTE_ARRAY
> ¯\_(ツ)_/¯
> ```
>
> and:
>
> ```text
> binary payload
> + spatial logical type
> + CRS contract
> ```
>
> **The breakthrough is not that the polygon survives disk. The meaning survives with it.**


## ✅ What This Run Established

```text
same WKB stored as BINARY and GEOMETRY(4326)        ✅
both use binary Parquet physical payload             ✅
plain BINARY has no spatial logical annotation       ✅
native Geometry has a spatial logical annotation     ✅
footer/annotation CRS evidence is inspected          ✅
Spark reads plain file back as BINARY                ✅
Spark reads native file back as GEOMETRY(4326)       ✅
WKB survives both round trips                        ✅
GeoParquet `geo` metadata absent                      ✅ runtime-proven
native GeospatialStatistics emitted                  ✅ runtime-proven
native bbox emitted                                  ✅ runtime-proven
bbox matches source geometries exactly               ✅ rerun assertion added
GEOMETRY(ANY) Parquet persistence is rejected        ✅
full spatial predicate pushdown                      ❌ not claimed
every GeoParquet convention automatically emitted   ❌ not claimed
```

The important practical result is simple:

> **With ordinary binary, the shape survives. With native spatial Parquet, the shape and its spatial contract can survive together.**

## 🚀 Wormhole Verdict

```text
BINARY
→ WKB survives


GEOMETRY(4326)
→ WKB survives
→ spatial logical type survives
→ CRS survives
→ Spark reconstructs GEOMETRY(4326)
→ native GeospatialStatistics survive
→ native bounding box survives
```

And the metadata layers remained distinct:

```text
Parquet GEOMETRY annotation   ✅
GeoParquet-style `geo` key    ❌
```

The fixed-SRID storage boundary remained intact:

```text
GEOMETRY(4326) → Parquet ✅
GEOMETRY(ANY)  → Parquet 💥
```

> **We proved Spark wrote the bbox. We did not prove Spark uses it to skip data.**

The storage layer has handed the execution engine ammunition.

Whether Spark actually fires it belongs in the next lab.

# 🛰️ Mission Handoff

We have now followed spatial meaning through:

```text
Python / SQL
    ↓
Catalyst
    ↓
Parquet
    ↓
disk
    ↓
Parquet
    ↓
Spark again
```

The nouns survive the wormhole.

The next question is where this metadata becomes **work avoided**:

> Can bounding boxes, partitioning, candidate pruning, and spatial execution turn “I know what this column is” into “I don't need to compare everything with everything”? 🌌⚔️

---

## 📚 Primary references

- Apache Spark — native geospatial types and Parquet persistence  
  https://spark.apache.org/docs/latest/sql-ref-geospatial-types.html

- Apache Parquet — native geospatial logical types  
  https://parquet.apache.org/docs/file-format/types/geospatial/

- Apache Parquet — native geospatial types announcement  
  https://parquet.apache.org/blog/2026/02/13/native-geospatial-types-in-apache-parquet/

- GeoParquet specification  
  https://geoparquet.org/

The notebook treats optional metadata as an empirical question: **inspect the file first, claim second.**

In [13]:
# Clean up the temporary Parquet datasets after all evidence has been captured.
spark.stop()

try:
    shutil.rmtree(workdir)
    cleanup = "deleted"
except Exception as exc:
    cleanup = f"could not delete automatically: {type(exc).__name__}: {exc}"

print(f"🛰️ Spark stopped. Orbit directory cleanup: {cleanup}")

🛰️ Spark stopped. Orbit directory cleanup: deleted
